# 05 - Conceptos Básicos de RAG (Retrieval Augmented Generation)

RAG es una técnica que combina:
1. **Retrieval**: Buscar documentos relevantes
2. **Augmented**: Aumentar el prompt con esos documentos
3. **Generation**: Generar respuesta usando LLM

Beneficios:
- Respuestas más precisas basadas en documentos reales
- Reduce alucinaciones del LLM
- Permite usar información actualizada
- No requiere reentrenar el modelo

## PARTE 1: Concepto de RAG vs LLM sin RAG

### Sin RAG: LLM genera respuesta de memoria

In [ ]:
import os
import sys
from dotenv import load_dotenv

sys.path.insert(0, '../src')
load_dotenv(dotenv_path="../.env")

from langchain_google_genai import ChatGoogleGenerativeAI

api_key = os.getenv("GOOGLE_API_KEY")

# LLM sin RAG: solo usa conocimiento interno
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.7
)

print("SIN RAG: Pregunta directa al LLM")
print("=" * 70)

pregunta = "Cual es la política de devoluciones de MiTienda?"
respuesta = llm.invoke(pregunta)

print(f"Pregunta: {pregunta}")
print(f"\nRespuesta (basada en conocimiento general):")
print(respuesta.content[:300] + "...")
print("\nProblema: No tiene acceso a documentos específicos de MiTienda")

### Con RAG: Busca en documentos primero, luego responde

In [ ]:
# CON RAG: Busca en documentos + pregunta al LLM

print("CON RAG: Búsqueda + Generación")
print("=" * 70)

# 1. Base de documentos (simulado)
documentos = [
    "Política de Devoluciones MiTienda: Los clientes pueden devolver productos dentro de 30 días de la compra en perfecto estado. Se reembolsa el 100% del monto pagado.",
    "Envios: Ofrecemos envio gratuito en compras mayores a $50. El tiempo de entrega es 3-5 dias habiles.",
    "Garantía: Todos nuestros productos tienen garantía de 1 año por defectos de fabrica."
]

# 2. Simular búsqueda (buscar documentos relevantes)
pregunta = "Cual es la política de devoluciones de MiTienda?"
documento_relevante = documentos[0]  # Simulado: busca el documento más relevante

print(f"Pregunta: {pregunta}")
print(f"\n1. RETRIEVAL (Búsqueda):")
print(f"   Documento encontrado: {documento_relevante[:80]}...")

# 3. Aumentar el prompt con el documento
prompt_con_rag = f"""Basándote en este documento:
{documento_relevante}

Responde la pregunta: {pregunta}"""

print(f"\n2. AUGMENTED (Aumento del prompt):")
print(f"   Prompt enviado al LLM incluye el documento")

# 4. Generar respuesta
respuesta_rag = llm.invoke(prompt_con_rag)

print(f"\n3. GENERATION (Generación):")
print(f"   Respuesta: {respuesta_rag.content[:300]}")
print(f"\nVentaja: Respuesta precisa basada en documentos reales")

## PARTE 2: Componentes de RAG

### Diagrama de RAG

```
┌─────────────┐
 │  Pregunta   │
 └──────┬──────┘
        │
        v
 ┌─────────────────────┐
 │  1. RETRIEVAL       │
 │  (Búsqueda)         │
 │  Vector Search      │
 │  Semantic Search    │
 └──────────┬──────────┘
            │
            v
 ┌─────────────────────┐
 │  Documentos         │
 │  Relevantes         │
 │  Top-K Results      │
 └──────────┬──────────┘
            │
            v
 ┌─────────────────────┐
 │  2. AUGMENTED       │
 │  Aumentar Prompt    │
 │  Pregunta +         │
 │  Documentos         │
 └──────────┬──────────┘
            │
            v
 ┌─────────────────────┐
 │  3. GENERATION      │
 │  LLM (Gemini)       │
 │  Respuesta          │
 └──────────┬──────────┘
            │
            v
      ┌──────────┐
      │ Respuesta│
      └──────────┘
```

### Componentes principales

In [ ]:
print("COMPONENTES DE RAG")
print("=" * 70)
print()

print("1. DOCUMENT STORE (Base de Documentos)")
print("   - Almacena documentos/textos a buscar")
print("   - Opciones: PDF, Web, Database, Archivos")
print()

print("2. RETRIEVER (Motor de Búsqueda)")
print("   - Busca documentos relevantes a la pregunta")
print("   - Tipos:")
print("     * Vector Search: Similaridad de embeddings")
print("     * Keyword Search: BM25, Lucene")
print("     * Hybrid: Combinación de ambos")
print()

print("3. EMBEDDINGS (Representación Vectorial)")
print("   - Convierte texto en vectores numéricos")
print("   - Permite buscar por similaridad semántica")
print("   - Ejemplos: OpenAI Embeddings, Gemini Embeddings")
print()

print("4. VECTOR STORE (Base de Datos Vectorial)")
print("   - Almacena embeddings para búsqueda rápida")
print("   - Opciones: FAISS, Pinecone, Chroma, Milvus")
print()

print("5. LLM (Modelo de Lenguaje)")
print("   - Genera respuesta basada en documentos + pregunta")
print("   - Ejemplo: Gemini 2.5 Flash")
print()

print("6. PROMPT TEMPLATE")
print("   - Estructura para combinar documentos + pregunta")
print("   - Define cómo se presenta la información al LLM")

## PARTE 3: RAG Simple con LangChain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.text_splitter import CharacterTextSplitter

print("RAG SIMPLE: Implementación Básica")
print("=" * 70)
print()

# PASO 1: Crear base de documentos
print("PASO 1: Document Store")
documentos_texto = """
Python es un lenguaje de programación interpretado, de alto nivel y orientado a objetos.
Se creó en 1991 por Guido van Rossum. Python es conocido por su simplicidad y legibilidad.

LangChain es un framework para desarrollar aplicaciones con LLMs.
Permite conectar modelos de lenguaje con APIs externas y bases de datos.
LangChain soporta múltiples proveedores: OpenAI, Google, Anthropic, etc.

RAG (Retrieval Augmented Generation) mejora las respuestas del LLM
al proporcionar documentos relevantes como contexto.
"""

# PASO 2: Dividir en chunks
print("PASO 2: Split Documents en chunks")
splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_text(documentos_texto)
print(f"   Documentos divididos en {len(chunks)} chunks")
print(f"   Chunk 1: {chunks[0][:100]}...")
print()

# PASO 3: Simular búsqueda
print("PASO 3: Retrieval (Buscar documentos relevantes)")
pregunta = "Que es RAG?"
# En sistema real usarías vector search. Aquí simulamos:
docs_recuperados = [c for c in chunks if 'RAG' in c or 'Retrieval' in c]
print(f"   Pregunta: {pregunta}")
print(f"   Documentos encontrados: {len(docs_recuperados)}")
print(f"   Contenido: {docs_recuperados[0][:100]}...")
print()

In [ ]:
# PASO 4: Crear prompt con RAG
print("PASO 4: Augmented Prompt")

rag_prompt = PromptTemplate.from_template(
    """Basándote en este contexto, responde la pregunta.

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta (basada en el contexto):"""  
)

contexto = "\n".join(docs_recuperados)
prompt_final = rag_prompt.format(contexto=contexto, pregunta=pregunta)

print("   Prompt construido:")
print("   " + prompt_final[:150] + "...")
print()

# PASO 5: Generar respuesta
print("PASO 5: Generation (Respuesta del LLM)")
respuesta = llm.invoke(prompt_final)
print(f"   {respuesta.content[:200]}...")

## PARTE 4: RAG Chain en LangChain

In [ ]:
from langchain.chains import RetrievalQA
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

print("RAG CHAIN: Usando LangChain RetrievalQA")
print("=" * 70)
print()

# Crear embeddings
print("1. Crear embeddings")
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=api_key
)
print("   Embeddings creados con Gemini")

# Crear vector store
print("\n2. Crear Vector Store")
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_text(documentos_texto)
vector_store = FAISS.from_texts(docs, embeddings)
print(f"   Vector Store creado con {len(docs)} documentos")

# Crear retriever
print("\n3. Crear Retriever")
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print("   Retriever configurado (top-2 resultados)")

# Crear RAG chain
print("\n4. Crear RAG Chain")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=False
)
print("   RAG Chain creado")

# Probar
print("\n5. Ejecutar consulta")
pregunta = "Cual es la diferencia entre RAG y LLM normal?"
respuesta = qa_chain.run(pregunta)
print(f"\nPregunta: {pregunta}")
print(f"\nRespuesta: {respuesta[:300]}...")

## PARTE 5: Comparación de Métodos

In [ ]:
print("COMPARACIÓN: RAG vs Sin RAG vs Fine-Tuning")
print("=" * 70)
print()

comparison = {
    "Sin RAG (LLM puro)": {
        "Velocidad": "Muy rápido",
        "Exactitud": "Media-Baja (alucinaciones)",
        "Datos actualizados": "No (conocimiento estático)",
        "Implementación": "Simple",
        "Costo": "Solo LLM API calls",
        "Casos de uso": "Preguntas generales, bots simples"
    },
    "RAG (Recomendado)": {
        "Velocidad": "Rápido (búsqueda + generación)",
        "Exactitud": "Alta (basado en documentos)",
        "Datos actualizados": "Si (actualiza documentos)",
        "Implementación": "Moderada (vector DB + LLM)",
        "Costo": "Vector search + LLM API calls",
        "Casos de uso": "QA sobre documentos, customer support"
    },
    "Fine-Tuning": {
        "Velocidad": "Rápido",
        "Exactitud": "Muy alta (modelo especializado)",
        "Datos actualizados": "No (requiere reentrenar)",
        "Implementación": "Compleja (datos + entrenamiento)",
        "Costo": "Alto (tokens + entrenamiento)",
        "Casos de uso": "Tareas muy específicas, producción"
    }
}

for metodo, caracteristicas in comparison.items():
    print(f"\n{metodo}")
    print("-" * 50)
    for aspecto, valor in caracteristicas.items():
        print(f"  {aspecto:20} -> {valor}")

## PARTE 6: Vector Stores Disponibles

In [ ]:
print("VECTOR STORES POPULARES EN LANGCHAIN")
print("=" * 70)
print()

vector_stores = {
    "FAISS": {
        "Descripcion": "In-memory, local, rápido",
        "Instalacion": "pip install faiss-cpu o faiss-gpu",
        "Casos de uso": "Prototipado, desarrollo local",
        "Ventajas": "Gratis, sin servidor, rápido",
        "Desventajas": "No persiste en disco fácilmente"
    },
    "Chroma": {
        "Descripcion": "Vector DB embebida, simple",
        "Instalacion": "pip install chromadb",
        "Casos de uso": "Prototipos, apps pequeñas",
        "Ventajas": "Fácil de usar, embebida",
        "Desventajas": "Limitada para producción"
    },
    "Pinecone": {
        "Descripcion": "Cloud vector DB escalable",
        "Instalacion": "pip install pinecone-client",
        "Casos de uso": "Producción, escala",
        "Ventajas": "Escalable, managed",
        "Desventajas": "Requiere API key, costo"
    },
    "Milvus": {
        "Descripcion": "Open-source, distribuido",
        "Instalacion": "Docker + cliente Python",
        "Casos de uso": "Producción open-source",
        "Ventajas": "Open source, escalable",
        "Desventajas": "Setup complejo"
    },
    "Weaviate": {
        "Descripcion": "Vector DB con API GraphQL",
        "Instalacion": "Docker o cloud",
        "Casos de uso": "Búsqueda semántica avanzada",
        "Ventajas": "Potente, flexible",
        "Desventajas": "Curva de aprendizaje"
    }
}

for nombre, detalles in vector_stores.items():
    print(f"\n{nombre}")
    print("-" * 50)
    for campo, valor in detalles.items():
        print(f"  {campo:20} -> {valor}")

## PARTE 7: Casos de Uso Prácticos

In [ ]:
print("CASOS DE USO DE RAG")
print("=" * 70)
print()

casos = [
    {
        "Nombre": "Customer Support Chatbot",
        "Descripcion": "Responder preguntas basado en documentos de soporte",
        "Documentos": "FAQs, políticas, guías de usuario",
        "Vector Store": "FAISS, Chroma",
        "Beneficio": "Respuestas consistentes, reduce carga de soporte"
    },
    {
        "Nombre": "Legal Document QA",
        "Descripcion": "Analizar contratos y leyes",
        "Documentos": "PDFs legales, jurisprudencia",
        "Vector Store": "Pinecone, Milvus",
        "Beneficio": "Búsqueda rápida en miles de documentos"
    },
    {
        "Nombre": "Medical Knowledge Base",
        "Descripcion": "Consultas médicas sobre literatura científica",
        "Documentos": "Papers científicos, protocolos médicos",
        "Vector Store": "Pinecone, Weaviate",
        "Beneficio": "Diagnósticos asistidos, evidencia basada"
    },
    {
        "Nombre": "Code Assistant",
        "Descripcion": "Responder preguntas sobre base de código",
        "Documentos": "Archivos fuente, documentación",
        "Vector Store": "FAISS, Chroma",
        "Beneficio": "Onboarding más rápido para desarrolladores"
    },
    {
        "Nombre": "Product QA",
        "Descripcion": "Preguntas sobre especificaciones de productos",
        "Documentos": "Catálogos, especificaciones, reviews",
        "Vector Store": "Chroma, Pinecone",
        "Beneficio": "Información actualizada automáticamente"
    }
]

for i, caso in enumerate(casos, 1):
    print(f"\n{i}. {caso['Nombre']}")
    print("-" * 50)
    for campo, valor in caso.items():
        if campo != 'Nombre':
            print(f"   {campo:20} -> {valor}")

## RESUMEN: Flujo Completo de RAG

In [ ]:
print("FLUJO COMPLETO DE RAG EN LANGCHAIN")
print("=" * 70)
print()

print("FASE 1: INDEXACIÓN (Una sola vez)")
print("-" * 70)
print("1. Cargar documentos")
print("   from langchain.document_loaders import PDFLoader")
print("   docs = PDFLoader('archivo.pdf').load()")
print()
print("2. Dividir en chunks")
print("   from langchain.text_splitter import CharacterTextSplitter")
print("   chunks = CharacterTextSplitter(...).split_documents(docs)")
print()
print("3. Crear embeddings")
print("   from langchain_google_genai import GoogleGenerativeAIEmbeddings")
print("   embeddings = GoogleGenerativeAIEmbeddings(...)")
print()
print("4. Guardar en vector store")
print("   from langchain_community.vectorstores import FAISS")
print("   vector_store = FAISS.from_documents(chunks, embeddings)")
print()
print()

print("FASE 2: RETRIEVAL-GENERATION (Cada pregunta)")
print("-" * 70)
print("1. Usuario pregunta")
print("   pregunta = 'Cual es tu política de devoluciones?'")
print()
print("2. Buscar documentos relevantes")
print("   docs_relevantes = vector_store.similarity_search(pregunta, k=3)")
print()
print("3. Crear prompt aumentado")
print("   prompt = f'Contexto:\n{docs_relevantes}\n\nPregunta: {pregunta}'")
print()
print("4. Generar respuesta")
print("   respuesta = llm.invoke(prompt)")
print()
print("5. Retornar respuesta")
print("   return respuesta")
print()
print()

print("VENTAJAS CLAVE DE RAG")
print("-" * 70)
print("✓ Respuestas basadas en documentos reales (no alucinaciones)")
print("✓ Fácil de actualizar documentos (sin reentrenamiento)")
print("✓ Explainabilidad (puedes mostrar qué documento se usó)")
print("✓ Escalable (agregar documentos es fácil)")
print("✓ Costo-efectivo (vs fine-tuning)")